# Incucyte Morphology Pipeline: Cells & Nuclei (Notebook)

**Must run in environment with GPU**

**Use environment_morphSegment.yml**

**Use incucyte_pipeline.py**

**Images Required for Each Sample Example (in one folder):**
- `KH2506_PHASE_F7_B6_1_01d00h00m.tif`  (8-bit phase)
- `KH2506_GFP_F7_B6_1_01d00h00m.tif`    (32-bit calibrated GFP)
- `KH2506_MC_F7_B6_1_01d00h00m.tif`     (32-bit calibrated mCherry)
- One **Designer** export with a **scale bar** at original resolution, e.g. `KH2506_OVERLAP_F7_B6_1_01d00h00m.tif`

**Filename format:** `<root>_<chan>_<sample>_<well>_<site>_<time>.tif`  
Example: `KH2506_GFP_F7_B6_1_01d00h00m.tif` where `F7 = sample`, `B6 = well`.

---

## Typical workflows
```python
### A) One sample, fully automatic (picks 50, saves QC+CSVs)
- Edit CONFIG as needed

### A) One sample, fully automatic (picks 50, saves QC+CSVs)
res = run_single_sample("F7")

### B) Interactive review (5-at-a-time + replace)
session = build_review_session("F7")
show_batch(session, start=0, count=5)
session = replace_selected(session, bad_indices=[3,5])
show_batch(session, start=0, count=5)
# ... iterate ...
finalize_review(session)

### C) Queue of samples (processed one-by-one)
res_all = run_queue(["F7","C5","B6"])

### Outputs go to: outputs/<sample>/ (CSVs, QC panel PNGs, masks).

## Edit Config
- Can look in incucyte_pipeline.py for additional config parameters.
- Settings as is work well on cells tested.

In [ ]:
##### SIMPLIFIED CONFIG ##############
from importlib import reload
import incucyte_pipeline as ip
reload(ip)

CONFIG = ip.PipelineConfig(
    parent_dir="./inputs",          # folder with your TIFs
    outputs_dir="./outputs",  # results root (per-sample folders inside)
    # use_selection_manifest_path="/home/kennedy/work/KH2506_morphology/outputs/F2/final/F2_selection_manifest.jsonn",  # adjust path/sample


    # If you know the scale exactly, set microns_per_pixel and skip bar detection:
    microns_per_pixel=None,   # e.g., 1.24

    # Designer bar length (used when microns_per_pixel is None)
    scale_bar_microns=800.0,
    
    # Edit size filters
    min_cell_area_um2 = 0.0,
    max_cell_area_um2 = 1e12,
    min_nuc_area_um2  = 0.0,
    max_nuc_area_um2  = 1e12,

    phase_gaussian_sigma=1.0,
    phase_thresh_method="otsu",
    phase_min_size_px=200,

    nuclei_gaussian_sigma=1.0,
    nuclei_min_size_px=10.0,
    nuclei_watershed=True,
    nuclei_watershed_compactness=0.00, #Higher compactness pulls borders in, yielding slightly smaller, rounder nuclei.

    # #nucleus settings
    nuc_top_hat_radius = 9, #higher = stronger background removal
    nuc_clahe_clip = 0.008, #**********0.01 works pretty well, higher boosts faint nuclei contrast a bit more.
    nuc_log_sigma_min = 1.2, #this and the max below widen the seed size range so both tiny and slightly larger blobs seed.
    nuc_log_sigma_max = 4.2,
    nuc_log_threshold = 0.018, #lower number lowers the LoG seed cutoff = more candidate nuclei. #.018 works good
    refine_nuclei_within_cell = True,
    subpixel_smooth_sigma = 0.0,   # set 0 to disable
    nuc_threshold_offset = 0.0, #Higher = stricter = smaller masks.
    nuc_sauvola_window = 25,
    nuc_sauvola_k = 0.2,
    nuc_min_pixels_after_thresh = 50,
    nuc_shrink_px = 0,
    nuc_fill_holes_area = 128,        # fill small interior holes
    nuc_open_radius = 1,              # 0 disables; 1 is a gentle trim
    # seeding / splitting
    nuc_min_distance_px = 6,          # **********fallback peak spacing (prevents merges)

    # Edit number of images to be analyzed (should be 1 unless more wells were added to folder)
    max_fovs_per_sample=1,
    target_cells=50,

    # Save selections
    save_individual_qc_panels=False,   # turn OFF per-cell PNGs
    save_combined_qc_panel=True,       # keep the big gallery
    save_selection_manifest=True,      # write the JSON manifest
    save_selection_maps=True           # (optional) per-FOV maps with red boxes
)

CONFIG
reload(ip)

## One-sample automatic run
- Change sample name one at a time to complete for each sample.

In [ ]:
res = ip.process_sample("F2", CONFIG)
res

## Interactive review (5-at-a-time + replacement)
- Comment out lines as you move through review. First just create the session. Then, comment out that line and show a batch. Comment out the batch and replace selected as needed. Then, comment out the replacement and show batch again, etc. When all samples are optimal, comment out all other lines and finalize the review.

In [ ]:
# 1) build the session (uses current CONFIG, but does NOT write CSVs yet)
# session = ip.build_review_session("F2", CONFIG)

# 2) page through in batches of 5
# ip.show_batch(session, start=0, count=5)      # cells 1–5
# ip.show_batch(session, start=5, count=5)    # cells 6–10, etc.

# 3) swap any wrong picks (1-based indices in the current global selection)
# session = ip.replace_selected(session, bad_indices=[])

# 4) re-check that batch
ip.show_batch(session, start=0, count=50)
# ip.show_batch(session, start=9, count=1)

# 5) finalize and write outputs using the CURRENT selection
final = ip.finalize_review(session)
final